# Tennis Data

##### Data Preprocessing

Pay attention to the relative path of data

In [5]:
import os
import pandas as pd


current_path = os.getcwd()
data_path = current_path + '\\' + 'data' + '\\' + 'tennis' + '\\' 
covariate = ['winner_age', 'loser_age']
df = combined_df = pd.DataFrame()
for file in os.listdir(data_path):
    if 'atp_matches' in file:
        tem = pd.read_csv(data_path+file,)
        tem = tem.drop(tem[tem['score'] == 'W/O'].index)
        tem = tem.dropna(subset=covariate)
        df = pd.concat([df, tem], ignore_index=True)
    else:
        pass

Select suitable matches and players <br>
<span style="font-size: smaller;">
We remove players whose competition times are less than 10 or who have never won.
</span> 

In [6]:
def delete_players(df,d_min):
    players = df[['winner_name','loser_name']]
    counts = pd.Series(players.values.ravel()).value_counts()
    ### d >= d_min
    remain_players = counts[counts>=d_min].index
    ### winner
    remain_players = remain_players[remain_players.isin(df['winner_name'])].tolist()
    n = len(remain_players)
    return remain_players,n
def delete_matches(df,remain_players):
    df = df[df[['winner_name','loser_name']].isin(remain_players).all(axis=1)]
    N = len(df)
    return df,N
def select(df,d_min=10):
    players = df[['winner_name','loser_name']]
    counts = pd.Series(players.values.ravel()).value_counts()
    all_players = counts.index.tolist()
    nn = len(all_players)
    NN = len(df)
    while True:
        print('-'*10)
        print(f'The number of matches: {NN}')
        print(f'The number of players: {nn}')
        remain_players,n = delete_players(df,d_min)
        df,N = delete_matches(df,remain_players)
        if NN == N and nn == n:
            break
        else:
            NN = N
            nn = n
    return df,n,N    
df,n,N = select(df,10)

----------
The number of matches: 145658
The number of players: 4603
----------
The number of matches: 138397
The number of players: 1991
----------
The number of matches: 137392
The number of players: 1856
----------
The number of matches: 137127
The number of players: 1825
----------
The number of matches: 137043
The number of players: 1815
----------
The number of matches: 136989
The number of players: 1809
----------
The number of matches: 136962
The number of players: 1806
----------
The number of matches: 136944
The number of players: 1804
----------
The number of matches: 136935
The number of players: 1803


In [7]:
df.to_csv(data_path+'tennis(preprocessed).csv',index=None)

##### Data Analysis

Model selection

In [1]:
import os
import pandas as pd
import numpy as np

# read data
current_path = os.getcwd()
data_path = current_path + '\\' + 'data' + '\\' + 'tennis' + '\\' 
df = pd.read_csv(data_path+'tennis(preprocessed).csv')


# players ID
players = df[['winner_name','loser_name']]
counts = pd.Series(players.values.ravel()).value_counts()
playerID = {value: index for index, value in enumerate(counts.index.tolist())}
n = len(playerID)
# matches
name_columns = ['winner_name','loser_name']
Matches = df[name_columns].replace(playerID)
T = np.array(Matches).tolist()
N = len(T)
# age
age_columns = ['winner_age','loser_age']
Age = np.array(df[age_columns])


C:\Users\13299\AppData\Local\Temp\ipykernel_6568\2923757027.py:8: DtypeWarning: Columns (8,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path+'tennis(preprocessed).csv')
C:\Users\13299\AppData\Local\Temp\ipykernel_6568\2923757027.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Matches = df[name_columns].replace(playerID)


In [2]:
# kernel
A = [25,30,35]
Lamb = [0.01, 0.03]
gauss_kernel = lambda x: np.array([np.exp(-lamb*(x-a)**2) for a in A for lamb in Lamb])
cov = [gauss_kernel(age).T for age in Age]

In [12]:
from package import algorithm

# BIC
def get_BIC(T,cov,s,N,n):
    X = [x[:,s] for x in cov]
    d = len(s)
    print(d)
    u_plusDC,v_plusDC = algorithm.AM(T,X,n,d,type = 'pair',detail=True)
    likelihood = algorithm.multi_likelihood(T,X,u_plusDC,v_plusDC)
    BIC = (n-1+d) * np.log(N) - 2*likelihood*N
    return BIC

In [ ]:
import itertools

# get_subset
get_subset = lambda n: [subset for i in range(n + 1)
                         for subset in itertools.combinations(list(range(n)), i)]
subset = get_subset(6)

In [16]:
from joblib import Parallel, delayed

res = Parallel(n_jobs=7)(delayed(get_BIC)(T,cov,subset[i],N,n) for i in range(len(subset)))
print('selected subset:', subset[np.argmin(res)])

TypeError: get_BIC() missing 4 required positional arguments: 'cov', 's', 'N', and 'n'